# Mercado de la vivienda en la CDMX

Análisis reproducible de demanda, oferta, precios y perfil sociodemográfico por
alcaldía (y camino a colonia/CP) de la Ciudad de México.

**Fuentes:** PADIM (portales: inmuebles24, vivanuncios, propiedades_com, mls,
doorvel) · SNIIV/SEDATU (déficit, rezago y financiamientos) · SEDUVI (suelo) ·
CONAPO 2020 (marginación por colonia y AGEB) · SEPOMEX (CP↔colonia) ·
INEGI (Censo 2020, ENIGH 2022, INPC) · CONEVAL (pobreza municipal) ·
SHF (índice de precios, manual).

Pipeline: `main.py --refresh` (descarga) → `src/build.py` (paneles) →
`src/derive.py` (derivados) → `src/analyze.py` (diagnóstico). Este notebook
replica el diagnóstico y deja las tablas listas para análisis propios.


In [1]:
import os
import sys
from pathlib import Path

# El kernel puede arrancar en notebooks/ o en la raíz del proyecto
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd() / "src"))

import analyze
import build
import conapo
import coneval
import derive
import enigh
import geo
import inegi
import report
import sepomex
import padim
import sniiv
import pandas as pd


In [2]:
paneles = build.paneles_padim()
mediana = build.panel_alcaldias(paneles["portales_actual"])
mediana.query("operacion == 'venta'").sort_values("precio_m2_mediana", ascending=False)


,alcaldia,operacion,n,precio_mediana,precio_p10,precio_p90,precio_m2_mediana,m2_mediana
2,Cuajimalpa de Morelos,venta,999,2596000.0,173860.0,12805800.0,27786.885246,206.0
3,Cuauhtémoc,venta,3858,3493500.0,169970.0,11890900.0,27759.592823,188.0
4,Miguel Hidalgo,venta,4134,3464500.0,182950.0,15852200.0,25936.187877,195.0
7,Álvaro Obregón,venta,1704,3251000.0,169820.0,10479000.0,24363.875471,199.0
0,Benito Juárez,venta,4452,3741000.0,198020.0,9736200.0,22880.454704,195.0
1,Coyoacán,venta,761,3480000.0,202800.0,9635000.0,22820.512821,190.0
5,Tlalpan,venta,584,3681500.0,1171000.0,7645000.0,17828.265315,206.0
6,Xochimilco,venta,556,2924500.0,967000.0,6551500.0,14829.513624,196.5


## Financiamientos de vivienda (SNIIV)

Acciones y monto de crédito hipotecario por los principales organismos en CDMX.


In [3]:
fin = build.panel_financiamientos()
derive.serie_financiamientos(fin)


,anio_calendario,acciones,monto_mdp
0,2013,84866,24755.260933
1,2014,109193,32738.157293
2,2015,83111,33350.677384
3,2016,157464,38549.539384
4,2017,99437,40164.745640
5,2018,95957,40005.583768
6,2019,112946,45980.032020
7,2020,82701,46679.489462
8,2021,153595,64132.415533
9,2022,119144,67900.112959


## Perfil sociodemográfico (CONAPO 2020)

Marginación por colonia (con CP) y por AGEB; se resume por alcaldía.


In [4]:
colonia = conapo.marginacion_colonia()
conapo.marginacion_ageb()
marg = pd.read_csv("data/processed/marginacion_alcaldia.csv")
marg["share_alta_total"] = marg["share_Alto"] + marg["share_Muy_alto"]
marg.sort_values("share_alta_total", ascending=False)


,alcaldia,poblacion_2020,share_Muy_bajo,share_Bajo,share_Medio,share_Alto,share_Muy_alto,imn_promedio,share_alta_total
14,Xochimilco,4.177705e+05,0.050225,0.398768,0.385579,0.129531,0.035897,0.950847,0.165428
10,Milpa Alta,1.363287e+05,0.000000,0.113811,0.727003,0.159187,0.000000,0.945696,0.159187
11,Tlalpan,6.709706e+05,0.147140,0.581255,0.161095,0.098945,0.011565,0.958668,0.110510
8,La Magdalena Contreras,2.481588e+05,0.126515,0.541603,0.288121,0.043761,0.000000,0.957765,0.043761
7,Iztapalapa,1.836011e+06,0.025597,0.365854,0.572307,0.036242,0.000000,0.953276,0.036242
12,Tláhuac,3.818719e+05,0.027472,0.594727,0.355430,0.009878,0.012493,0.956808,0.022371
5,Gustavo A. Madero,1.154865e+06,0.125969,0.571737,0.293442,0.007284,0.001567,0.960195,0.008851
3,Cuajimalpa de Morelos,2.095128e+05,0.082736,0.595908,0.319835,0.001522,0.000000,0.960134,0.001522
15,Álvaro Obregón,7.585774e+05,0.204410,0.449063,0.345765,0.000763,0.000000,0.960848,0.000763
0,Azcapotzalco,4.321446e+05,0.228482,0.694638,0.076486,0.000393,0.000000,0.966387,0.000393


## Bridge colonia↔CP (SEPOMEX)

Catálogo de códigos postales de CDMX: cada CP con sus asentamientos y alcaldía.
- `data/processed/cp_cdmx.csv` — CP → asentamiento → alcaldía.
- `data/processed/conapo_marginacion_colonia.csv` — colonia → CP → indicadores.


In [5]:
import pandas as pd
cp = pd.read_csv("data/processed/cp_cdmx.csv")
cp.head(5)


,cp,asentamiento,tipo_asentamiento,alcaldia,zona
0,1000,San Ángel,Colonia,Álvaro Obregón,Urbano
1,1010,Los Alpes,Colonia,Álvaro Obregón,Urbano
2,1020,Guadalupe Inn,Colonia,Álvaro Obregón,Urbano
3,1030,Axotla,Pueblo,Álvaro Obregón,Urbano
4,1030,Florida,Colonia,Álvaro Obregón,Urbano


## Censo de Población y Vivienda 2020 (INEGI)

Principales resultados por AGEB y manzana urbana de la CDMX: 66,456 manzanas
y 2,211 AGEB con población y vivienda. Paneles:
- `censo2020_manzanas_cdmx.csv` — una fila por manzana (clave `cve_manzana`).
- `censo2020_ageb_cdmx.csv` — agregados por AGEB (cruza con CONAPO vía
  `censo/ageb_marginacion`).
- `censo2020_alcaldia.csv` — agregación por alcaldía con indicadores.


In [6]:
censo_al = pd.read_csv("data/processed/censo2020_alcaldia.csv")
censo_al[["alcaldia", "POBTOT", "indice_dependencia", "hacinamiento_prom",
          "vph_sin_agua_pct", "vph_piso_tierra_pct"]].round(2).sort_values(
    "indice_dependencia", ascending=False)


,alcaldia,POBTOT,indice_dependencia,hacinamiento_prom,vph_sin_agua_pct,vph_piso_tierra_pct
5,Gustavo A. Madero,1173351,52.44,3.50,0.17,0.13
13,Venustiano Carranza,443704,51.77,3.42,0.01,0.06
2,Coyoacán,614447,51.68,3.35,0.02,0.08
8,La Magdalena Contreras,246428,50.85,3.91,0.97,0.45
6,Iztacalco,404695,50.39,3.58,0.02,0.04
10,Milpa Alta,128710,50.36,4.07,4.41,1.20
0,Azcapotzalco,432205,49.75,3.32,0.12,0.07
15,Álvaro Obregón,759003,49.28,3.78,0.12,0.21
7,Iztapalapa,1835486,49.15,3.72,0.18,0.20
14,Xochimilco,429481,49.14,3.78,6.47,1.46


In [7]:
censo_ageb = pd.read_csv("data/processed/censo2020_ageb_marginacion.csv")
censo_ageb.shape


(2161, 10)

## Pobreza 2020 por alcaldía (CONEVAL)


In [8]:
coneval.pobreza_alcaldia()
pd.read_csv("data/processed/coneval_pobreza_alcaldia.csv").sort_values(
    "pobreza_porcentaje", ascending=False).head()


,alcaldia,municipio,poblacion,pobreza_porcentaje,pobreza_extrema_porcentaje,pobreza_moderada_porcentaje,vulnerable_carencias_sociales_porcentaje,vulnerable_ingresos_porcentaje,no_pobre_no_vulnerable_porcentaje,carencia_rezago_educativo_porcentaje,carencia_servicios_de_salud_porcentaje,carencia_seguridad_social_porcentaje,carencia_calidad_espacios_vivienda_porcentaje,carencia_servicios_basicos_vivienda_porcentaje,carencia_alimentacion_nutritiva_calidad_porcentaje,al_menos_una_carencia_porcentaje,al_menos_tres_carencias_porcentaje,ingreso_inferior_a_lpi_porcentaje,ingreso_inferior_a_lpei_porcentaje
7,Milpa Alta,Milpa Alta,161267,54.7,9.1,45.5,26.8,5.1,13.4,10.7,34.4,71.6,11.6,23.3,23.9,81.5,26.2,59.8,21.3
11,Xochimilco,Xochimilco,437383,48.2,9.3,38.9,21.9,8.7,21.3,9.7,31.4,58.7,10.5,13.0,19.0,70.1,19.4,56.9,22.9
5,Iztapalapa,Iztapalapa,1760382,43.9,6.6,37.3,22.2,10.5,23.4,10.2,31.8,54.3,6.3,0.6,20.3,66.1,13.6,54.4,20.6
6,La Magdalena Contreras,La Magdalena Contreras,241690,42.5,6.7,35.8,21.7,11.7,24.1,11.2,29.2,48.2,6.3,8.9,15.8,64.2,13.5,54.2,21.1
9,Tláhuac,Tláhuac,377744,42.4,5.1,37.3,23.3,9.3,25.0,9.1,26.0,54.9,6.2,4.1,18.0,65.7,12.1,51.7,17.6


## Ingresos de los hogares (ENIGH 2022) y asequibilidad

La ENIGH es muestral a nivel entidad: el ingreso se estima para toda la CDMX.
Se infla con el INPC (2022 → 2026-08). La asequibilidad cruza ese ingreso contra
la mediana del inventario en venta de cada alcaldía (años de ingreso para
comprar + % de hogares que solventan el 30% de la mensualidad).


In [9]:
enigh.ingresos_cdmx()
ing = pd.read_csv("data/processed/enigh_cdmx_ingresos.csv")
ing


,entidad,hogares_ponderados,ingreso_anual_mediano,ingreso_mensual_mediano,p25_mensual,p10_mensual,p75_mensual,inpc_factor_2022_2026
0,Ciudad de México,2990030,298020.54369,24835.045308,16185.240671,11095.110612,40229.923547,1.1874


In [10]:
hogares = pd.read_csv("data/processed/enigh_cdmx_hogares.csv")
mediana = build.panel_alcaldias(build.paneles_padim()["portales_actual"])
aseq = derive.asequibilidad(mediana, ing, hogares)
aseq.sort_values("anios_ingreso")


,alcaldia,precio_mediana,anios_ingreso,mensualidad_estimada,pct_ingreso_mensual,share_hogares_alcanza_30pct
2,Cuajimalpa de Morelos,2596000.0,8.7,23326.147679,93.9,0.078
6,Xochimilco,2924500.0,9.8,26277.857815,105.8,0.057
7,Álvaro Obregón,3251000.0,10.9,29211.597113,117.6,0.044
4,Miguel Hidalgo,3464500.0,11.6,31129.984066,125.3,0.038
1,Coyoacán,3480000.0,11.7,31269.258060,125.9,0.036
3,Cuauhtémoc,3493500.0,11.7,31390.561216,126.4,0.035
5,Tlalpan,3681500.0,12.4,33079.819985,133.2,0.031
0,Benito Juárez,3741000.0,12.6,33614.452414,135.4,0.029


## Notas de nivel geográfico

- `alcaldias_mediana.csv`: precio mediano y m² por alcaldía (venta/renta).
- PADIM entrega geografía G5 (zona de densidad), no predio exacto: el salto a
  colonia/CP exige unir esas zonas con las colonias del INEGI/DENUE.
- Los CP de CDMX no son polígonos: el *bridge* colonia↔CP se arma con SEPOMEX
  + CONAPO (ver `NOTES.md`).
